# 01: Configuration, Normalizer & Schema Validation

Interactive walkthrough for real estate domain config, Hinglish unit parsing, and Pydantic validation.

In [ ]:
# 1. Import config, normalizer, and schema modules
from src.config import load_domain_config
from src.normalizer import (
    clean_text,
    extract_bhk,
    extract_budget,
    normalize_city,
    parse_currency_amount,
)
from src.schema import RealEstateQuery

In [ ]:
# 2. Load the real estate domain configuration
config = load_domain_config("real_estate")
print(f"Domain: {config['domain']}")
print(f"Fields in schema: {list(config['schema'].keys())}")

In [ ]:
# 3. Inspect city aliases from YAML
aliases = config["city_aliases"]
print(f"Total city aliases configured: {len(aliases)}")
for raw, canonical in list(aliases.items())[:6]:
    print(f"  {raw:<15} -> {canonical}")

In [ ]:
# 4. Normalize raw city text
test_cities = ["gurgaon", "bombay", "bangalore", "noida"]
for city in test_cities:
    print(f"{city:<12} -> {normalize_city(city, aliases)}")

In [ ]:
# 5. Inspect unit multipliers
units = config["units"]
print("Configured unit multipliers:")
for u, mult in units.items():
    print(f"  {u:<10} -> {mult:>10,}")

In [ ]:
# 6. Test currency amount parsing
print(f"25k     -> INR {parse_currency_amount('25', 'k', units):,}")
print(f"1.5 cr  -> INR {parse_currency_amount('1.5', 'cr', units):,}")
print(f"80 lakh -> INR {parse_currency_amount('80', 'lakh', units):,}")

In [ ]:
# 7. Test budget extraction from Hinglish query patterns
budget_queries = ["1.5 cr tak", "under 25k", "25k to 30k", "starting 50 lakh", "5-8 cr"]
for q in budget_queries:
    b_min, b_max = extract_budget(q, units)
    print(f"{q:<20} -> min={b_min} | max={b_max}")

In [ ]:
# 8. Test BHK extraction
bhk_queries = ["noida 2bhk flat", "3 bhk ready to move", "1 rk studio"]
for q in bhk_queries:
    print(f"{q:<24} -> bhk={extract_bhk(q)}")

In [ ]:
# 9. Validate a search filter using Pydantic schema
filter_obj = RealEstateQuery(
    city="Gurugram",
    bhk=3,
    possession="ready_to_move",
    budget_max=15000000,
)
print("Validated search filter:")
print(filter_obj.model_dump(exclude_none=True))

In [ ]:
# 10. Verify range validation error handling
try:
    RealEstateQuery(budget_min=20000000, budget_max=10000000)
except ValueError as e:
    print(f"Caught expected range error: {e}")